# 08 — Observability and Tracing

**Learning objective:** record run identity, node lifecycle and latency, selected routes, retries, invocation counts, and termination reason for an executable graph.

A final answer cannot explain why a graph was slow, expensive, repeatedly retried, or terminated through a fallback. Observability makes execution policy inspectable.

## Mental model and topology

```mermaid
flowchart LR
    accTitle: Observable feedback graph
    accDescr: Generation and evaluation emit node spans while routing, retry, model calls, tool calls, and termination are recorded on one run trace.

    generate[Generate] --> evaluate[Evaluate]
    evaluate --> gate{Quality policy}
    gate -->|Low| improve[Improve]
    improve --> generate
    gate -->|Pass| complete([Complete])
```

## Trace contract and responsibilities

`RunTrace` owns a run ID, ordered `NodeTrace` events, routes, retry and call counters, usage, and termination. Nodes create spans around computation. Routers record selected branches. Retry and completion policy record their own decisions. The graph state remains business state; the trace is operational evidence.

In [1]:
from graph_engineering.observability import RunTrace, build_observed_graph

class StepClock:
    """A deterministic clock for reproducible lesson output."""
    def __init__(self):
        self.value = 0.0
    def __call__(self):
        self.value += 0.001
        return self.value

In [2]:
run_trace = RunTrace("demo-001", clock=StepClock())
graph = build_observed_graph(run_trace)
result = graph.invoke({"quality_scores": [0.5, 0.9], "trace": []})
print(run_trace.render())

Run: demo-001
START
  -> generate (attempt 1, 1.00 ms, ok)
  -> evaluate (attempt 1, 1.00 ms, ok)
  -> improve (attempt 1, 1.00 ms, ok)
  -> generate (attempt 2, 1.00 ms, ok)
  -> evaluate (attempt 2, 1.00 ms, ok)
  -> complete (attempt 1, 1.00 ms, ok)
  -> END
nodes visited: 6
retries: 1
llm calls: 2
tool calls: 1
termination: quality_reached


In [3]:
summary = run_trace.summary()
print("routes:", summary["routes"])
print("tokens:", summary["token_usage"])
print("final strategy:", result["strategy"])

routes: ['improve', 'complete']
tokens: 40
final strategy: add_evidence


## Failure considerations

Do not rely on timing assertions in ordinary tests; the deterministic clock is only for stable teaching output. Production traces need clock discipline, redaction, sampling, retention, and correlation with tool/provider IDs. A trace should record sanitized error types, not secrets, raw credentials, or unnecessary prompt data.

## What to modify

Change the score sequence to take the direct route, then inspect differences in nodes visited, calls, routes, and retries. Add one bounded failure and ensure the termination reason still explains the outcome.

**Next:** [09 — Subgraphs](09_subgraphs.ipynb) introduces intentional boundaries for larger systems.